# 01 — Data Ingestion
**Phishing URL Detection — CS 4200 Final Project**  
**Author:** Joe Casperson

**Purpose:**  
Load the raw CIC Phishing URL Dataset into PostgreSQL without any modifications.  
This notebook establishes the ground truth — the `urls` table holds the original  
data exactly as downloaded before any cleaning or preprocessing.

**Dataset:**  
- 79 pre-engineered numeric features per URL sample  
- Label column: `URL_Type_obf_Type` → encoded to `1` (phishing) / `0` (benign)

**Output:**  
- `urls` table populated in PostgreSQL  
- Row count verified against source CSV

---

## 1. Imports and Database Connection

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

POSTGRES_USER     = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB       = os.getenv('POSTGRES_DB')
POSTGRES_HOST     = os.getenv('POSTGRES_HOST', 'localhost')

conn_str = f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}/{POSTGRES_DB}'
engine   = create_engine(conn_str)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('✓ Connected to PostgreSQL')
    print(f'  {result.fetchone()[0]}')

✓ Connected to PostgreSQL
  PostgreSQL 16.13 (Ubuntu 16.13-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## 2. Load Raw CSV

In [2]:
CSV_PATH = '../data/raw/urls.csv'

df_raw = pd.read_csv(CSV_PATH)

print(f'✓ CSV loaded successfully')
print(f'  Rows    : {len(df_raw):,}')
print(f'  Columns : {len(df_raw.columns)}')
print(f'  Memory  : {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

✓ CSV loaded successfully
  Rows    : 15,367
  Columns : 80
  Memory  : 10.08 MB


In [ ]:
df_raw.head()

## 3. Column Mapping and Label Encoding

Two things happen here:
1. Column names are sanitized to match the PostgreSQL schema (lowercase, no special characters)
2. The text label `benign`/`phishing` is encoded to `0`/`1`

In [23]:
# Sanitize all column names to lowercase and remove special characters
# 'this.fileExtLen' → 'fileextlen'
# 'sub-Directory_LongestWordLength' → 'subdirectory_longestwordlength'
df = df_raw.copy()
df.columns = (
    df.columns
    .str.lower()
    .str.replace(r'[^a-z0-9_]', '_', regex=True)  # replace special chars with _
    .str.replace(r'_+', '_', regex=True)            # collapse multiple underscores
    .str.strip('_')                                  # strip leading/trailing underscores
    .str.replace(r'^this_', '', regex=True)   # handles 'this.fileExtLen' → 'fileextlen'
)
# Manual fix for hyphenated column name
df = df.rename(columns={'sub_directory_longestwordlength': 'subdirectory_longestwordlength'})

print('Column names after sanitization:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

Column names after sanitization:
   1. querylength
   2. domain_token_count
   3. path_token_count
   4. avgdomaintokenlen
   5. longdomaintokenlen
   6. avgpathtokenlen
   7. tld
   8. charcompvowels
   9. charcompace
  10. ldl_url
  11. ldl_domain
  12. ldl_path
  13. ldl_filename
  14. ldl_getarg
  15. dld_url
  16. dld_domain
  17. dld_path
  18. dld_filename
  19. dld_getarg
  20. urllen
  21. domainlength
  22. pathlength
  23. subdirlen
  24. filenamelen
  25. fileextlen
  26. arglen
  27. pathurlratio
  28. argurlratio
  29. argdomanratio
  30. domainurlratio
  31. pathdomainratio
  32. argpathratio
  33. executable
  34. isporteighty
  35. numberofdotsinurl
  36. isipaddressindomainname
  37. charactercontinuityrate
  38. longestvariablevalue
  39. url_digitcount
  40. host_digitcount
  41. directory_digitcount
  42. file_name_digitcount
  43. extension_digitcount
  44. query_digitcount
  45. url_letter_count
  46. host_letter_count
  47. directory_lettercount
  48. filename_l

In [24]:
# Encode the label column
# Original: 'phishing' / 'benign'  →  Encoded: 1 / 0
LABEL_COL = 'url_type_obf_type'    # name after sanitization

print(f'Label column raw values:')
print(df[LABEL_COL].value_counts())

df['label'] = df[LABEL_COL].map({'phishing': 1, 'benign': 0})
df = df.drop(columns=[LABEL_COL])

# Confirm encoding worked — should show 0 NaNs
print(f'\nEncoded label NaNs (should be 0): {df["label"].isna().sum()}')
print(f'Encoded value counts:')
print(df['label'].value_counts())

Label column raw values:
url_type_obf_type
benign      7781
phishing    7586
Name: count, dtype: int64

Encoded label NaNs (should be 0): 0
Encoded value counts:
label
0    7781
1    7586
Name: count, dtype: int64


## 4. First-Look Data Inspection

High-level audit of the raw data before it enters the database.  
**Screenshot this section** for the Chapter 9 'Before' state in your report.

In [25]:
print('── DataFrame Info ──────────────────────────────')
df.info()

── DataFrame Info ──────────────────────────────
<class 'pandas.DataFrame'>
RangeIndex: 15367 entries, 0 to 15366
Data columns (total 80 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   querylength                     15367 non-null  int64  
 1   domain_token_count              15367 non-null  int64  
 2   path_token_count                15367 non-null  int64  
 3   avgdomaintokenlen               15367 non-null  float64
 4   longdomaintokenlen              15367 non-null  int64  
 5   avgpathtokenlen                 15096 non-null  float64
 6   tld                             15367 non-null  int64  
 7   charcompvowels                  15367 non-null  int64  
 8   charcompace                     15367 non-null  int64  
 9   ldl_url                         15367 non-null  int64  
 10  ldl_domain                      15367 non-null  int64  
 11  ldl_path                        15367 non-null  int64  

In [26]:
print('── Descriptive Statistics (key features) ───────')
key_cols = ['label', 'urllen', 'domain_token_count', 'entropy_url',
            'symbolcount_url', 'numberrate_url', 'numberofdotsinurl']
df[key_cols].describe().round(4)

── Descriptive Statistics (key features) ───────


,label,urllen,domain_token_count,entropy_url,symbolcount_url,numberrate_url,numberofdotsinurl
count,15367.0000,15367.0000,15367.0000,15367.0000,15367.0000,15367.0000,15367.000
mean,0.4937,76.0354,2.5437,0.7217,7.1851,0.0861,2.362
std,0.5000,33.4490,0.9449,0.0492,3.5428,0.1005,1.781
min,0.0000,37.0000,2.0000,0.4196,2.0000,0.0000,1.000
25%,0.0000,54.0000,2.0000,0.6872,5.0000,0.0000,1.000
50%,0.0000,68.0000,2.0000,0.7232,6.0000,0.0566,2.000
75%,1.0000,90.0000,3.0000,0.7579,8.0000,0.1250,3.000
max,1.0000,348.0000,19.0000,0.8697,47.0000,0.7619,20.000


In [27]:
# Null counts
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
null_report = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
has_nulls   = null_report[null_report['null_count'] > 0]

print('── Null Values ─────────────────────────────────')
print(has_nulls if len(has_nulls) > 0 else '✓ No null values found')

── Null Values ─────────────────────────────────
                          null_count  null_pct
avgpathtokenlen                  271      1.76
numberrate_directoryname           9      0.06
numberrate_filename                9      0.06
numberrate_extension            7355     47.86
numberrate_afterpath               3      0.02
entropy_directoryname           1826     11.88
entropy_filename                 190      1.24
entropy_extension                  3      0.02
entropy_afterpath                  3      0.02


In [28]:
# Duplicate rows
dupe_count = df.duplicated().sum()
print(f'── Duplicate Rows ──────────────────────────────')
print(f'  Total duplicates : {dupe_count:,}')
print(f'  Pct of dataset   : {dupe_count / len(df) * 100:.2f}%')

── Duplicate Rows ──────────────────────────────
  Total duplicates : 544
  Pct of dataset   : 3.54%


In [29]:
# Class balance
label_counts = df['label'].value_counts().sort_index()
balance = pd.DataFrame({
    'label_name' : ['Benign (0)', 'Phishing (1)'],
    'count'      : label_counts.values,
    'pct'        : (label_counts.values / len(df) * 100).round(2)
})
print('── Class Balance ───────────────────────────────')
print(balance.to_string(index=False))

── Class Balance ───────────────────────────────
  label_name  count   pct
  Benign (0)   7781 50.63
Phishing (1)   7586 49.37


## 5. Align Columns with PostgreSQL Schema

Keep only the columns defined in `schema.sql` and put `label` first.

In [30]:
# Columns defined in schema.sql (excluding url_id and inserted_at — auto-generated)
SCHEMA_COLS = [
    'label',
    'querylength', 'domain_token_count', 'path_token_count',
    'avgdomaintokenlen', 'longdomaintokenlen', 'avgpathtokenlen',
    'tld', 'charcompvowels', 'charcompace',
    'ldl_url', 'ldl_domain', 'ldl_path', 'ldl_filename', 'ldl_getarg',
    'dld_url', 'dld_domain', 'dld_path', 'dld_filename', 'dld_getarg',
    'urllen', 'domainlength', 'pathlength', 'subdirlen',
    'filenamelen', 'fileextlen', 'arglen',
    'pathurlratio', 'argurlratio', 'argdomanratio',
    'domainurlratio', 'pathdomainratio', 'argpathratio',
    'executable', 'isporteighty', 'numberofdotsinurl',
    'isipaddressindomainname', 'charactercontinuityrate', 'longestvariablevalue',
    'url_digitcount', 'host_digitcount', 'directory_digitcount',
    'file_name_digitcount', 'extension_digitcount', 'query_digitcount',
    'url_letter_count', 'host_letter_count', 'directory_lettercount',
    'filename_lettercount', 'extension_lettercount', 'query_lettercount',
    'longestpathtokenlength', 'domain_longestwordlength', 'path_longestwordlength',
    'subdirectory_longestwordlength', 'arguments_longestwordlength',
    'url_sensitiveword', 'urlqueries_variable', 'spcharurl',
    'delimeter_domain', 'delimeter_path', 'delimeter_count',
    'numberrate_url', 'numberrate_domain', 'numberrate_directoryname',
    'numberrate_filename', 'numberrate_extension', 'numberrate_afterpath',
    'symbolcount_url', 'symbolcount_domain', 'symbolcount_directoryname',
    'symbolcount_filename', 'symbolcount_extension', 'symbolcount_afterpath',
    'entropy_url', 'entropy_domain', 'entropy_directoryname',
    'entropy_filename', 'entropy_extension', 'entropy_afterpath',
]

# Only keep columns that exist in both the schema list and the DataFrame
available_cols = [c for c in SCHEMA_COLS if c in df.columns]
missing_cols   = [c for c in SCHEMA_COLS if c not in df.columns]

df_load = df[available_cols].copy()

print(f'✓ Columns aligned: {len(available_cols)} matched')
if missing_cols:
    print(f'  Missing from CSV (will be NULL in DB): {missing_cols}')

✓ Columns aligned: 80 matched


## 6. Load into PostgreSQL

In [31]:
df_load.to_sql(
    name      = 'urls',
    con       = engine,
    if_exists = 'append',   # schema.sql already created the table
    index     = False,
    chunksize = 1000,        # write in batches for large datasets
)

print(f'✓ Loaded {len(df_load):,} rows into urls table')

✓ Loaded 15,367 rows into urls table


## 7. Verification

In [32]:
with engine.connect() as conn:
    pg_count = conn.execute(text('SELECT COUNT(*) FROM urls')).scalar()

csv_count = len(df_raw)

print('── Ingestion Verification ──────────────────────')
print(f'  CSV rows             : {csv_count:,}')
print(f'  PostgreSQL urls rows : {pg_count:,}')
print()

if pg_count == csv_count:
    print('✓ Row counts match — ingestion successful')
else:
    print(f'✗ Mismatch: {csv_count - pg_count:,} rows missing — check column alignment')

── Ingestion Verification ──────────────────────
  CSV rows             : 15,367
  PostgreSQL urls rows : 15,367

✓ Row counts match — ingestion successful


In [33]:
# Class balance in PostgreSQL — should match Section 4
balance_sql = '''
    SELECT
        label,
        CASE WHEN label = 1 THEN 'Phishing' ELSE 'Benign' END AS label_name,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM urls
    GROUP BY label
    ORDER BY label DESC
'''
print('── Class Balance in PostgreSQL ─────────────────')
display(pd.read_sql(balance_sql, engine))

── Class Balance in PostgreSQL ─────────────────


,label,label_name,count,pct
0,1,Phishing,7586,49.37
1,0,Benign,7781,50.63


In [34]:
# Quick peek at what landed in the database
print('── Sample rows from PostgreSQL ─────────────────')
sample_cols = 'url_id, label, urllen, domain_token_count, entropy_url, symbolcount_url'
display(pd.read_sql(f'SELECT {sample_cols} FROM urls LIMIT 5', engine))

── Sample rows from PostgreSQL ─────────────────


,url_id,label,urllen,domain_token_count,entropy_url,symbolcount_url
0,1,0,80.0,2.0,0.676804,6.0
1,2,0,78.0,3.0,0.715629,7.0
2,3,0,71.0,2.0,0.677701,8.0
3,4,0,64.0,2.0,0.696067,5.0
4,5,0,68.0,2.0,0.747202,9.0


---
## Summary

| Item | Value |
|---|---|
| Source file | `data/raw/urls.csv` |
| Raw rows loaded | *15,367* |
| Features loaded | 79 numeric columns |
| Label encoding | phishing → 1, benign → 0 |
| Duplicates detected | *544 (3.54%)* |
| Null values detected | *9 columns affect* |
| Table populated | `urls` |

**Next:** `02_preprocessing.ipynb` — audit, clean, and load into `clean_urls`.